In [ ]:

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModel, T5EncoderModel
import numpy as np
import os
from tqdm import tqdm
import gc
import json
import random
from datetime import datetime
warnings.filterwarnings('ignore')

print("=" * 80)
print("Loading datasets...")
print("=" * 80)

train_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/train_original.csv")
test_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/test_original.csv")
tag_vocab = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/unique_tags.csv")["Tag"].tolist()

print(f"✓ Training samples: {len(train_data)}")
print(f"✓ Testing samples: {len(test_data)}")
print(f"✓ Languages: {len(tag_vocab)}")
print(f"Languages: {tag_vocab}")

print("\n" + "=" * 80)
print("Loading CodeT5 tokenizer...")
print("=" * 80)

MODEL_NAME = "Salesforce/codet5-base"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f"✓ CodeT5 tokenizer loaded successfully!")
except Exception as e:
    print(f"✗ Error loading CodeT5: {e}")
    exit(1)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer config: pad_token={tokenizer.pad_token}")

def tokenize_batch(texts, max_length=256):
    """Tokenize code snippets with CodeT5"""
    return tokenizer(
        texts,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

print("\nTokenizing training data...")
train_texts = train_data["code"].tolist()
train_encodings = tokenize_batch(train_texts, max_length=256)

print("Tokenizing test data...")
test_texts = test_data["code"].tolist()
test_encodings = tokenize_batch(test_texts, max_length=256)

class CodeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.input_ids = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

class CodeT5Classifier(nn.Module):
    def __init__(self, num_classes, model_name=MODEL_NAME, device=None, freeze_layers=True):
        super().__init__()
        
        print(f"Loading {model_name}...")
        
    
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        
        # Get hidden size
        hidden_size = self.encoder.config.hidden_size
        print(f"  Hidden size: {hidden_size}")
        print(f"  Model type: {type(self.encoder)}")
        
        
        if freeze_layers:
            
            if hasattr(self.encoder, 'encoder'):
                encoder_module = self.encoder.encoder
                if hasattr(encoder_module, 'block'):
                    num_layers = len(encoder_module.block)
                    frozen_count = 0
                    
                    for i, layer in enumerate(encoder_module.block):
                        if i < num_layers // 2:  
                            for param in layer.parameters():
                                param.requires_grad = False
                            frozen_count += 1
                    print(f"  Frozen {frozen_count}/{num_layers} encoder blocks")
            
            
            if hasattr(self.encoder, 'shared'):
                for param in self.encoder.shared.parameters():
                    param.requires_grad = False
                print(f"  Frozen embeddings")
        
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )
        
        
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
        
        
        if device:
            self.encoder = self.encoder.to(device)
            self.classifier = self.classifier.to(device)
        
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        print(f"✓ CodeT5 Encoder model loaded successfully!")
        print(f"  Num classes: {num_classes}")
        print(f"  Total parameters: {total_params:,}")
        print(f"  Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
    
    def forward(self, input_ids, attention_mask):
        
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        
        last_hidden = outputs.last_hidden_state
        
        
        mask = attention_mask.unsqueeze(-1).float()
        sum_embeddings = torch.sum(last_hidden * mask, dim=1)
        sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
        pooled = sum_embeddings / sum_mask
        
        
        logits = self.classifier(pooled)
        return logits

def train_epoch(model, data_loader, optimizer, criterion, device, gradient_accumulation_steps=4):
    """Train for one epoch with gradient accumulation"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    optimizer.zero_grad()
    
    for step, batch in enumerate(tqdm(data_loader, desc="Training"), 1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss = loss / gradient_accumulation_steps
        loss.backward()
        
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        if step % gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * gradient_accumulation_steps
        
        if step % 20 == 0:
            avg_loss = total_loss / step
            accuracy = correct / total
            tqdm.write(f"Step {step}/{len(data_loader)} - Loss: {avg_loss:.4f}, Acc: {accuracy:.4f}")
    
    if len(data_loader) % gradient_accumulation_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
    
    return total_loss / len(data_loader), correct / total

def evaluate_model(model, data_loader, device, tag_vocab):
    """Evaluate model and return detailed results"""
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            
            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels)
            all_probs.extend(probs.cpu().numpy())
            
            if len(all_preds) % 200 == 0:
                accuracy_so_far = accuracy_score(all_labels, all_preds)
                tqdm.write(f"  Evaluated {len(all_preds)} samples - Accuracy: {accuracy_so_far:.4f}")
    
    accuracy = accuracy_score(all_labels, all_preds)
    
    per_lang_results = {}
    for i, lang in enumerate(tag_vocab):
        lang_indices = np.where(np.array(all_labels) == i)[0]
        if len(lang_indices) > 0:
            lang_acc = accuracy_score(
                np.array(all_labels)[lang_indices],
                np.array(all_preds)[lang_indices]
            )
            per_lang_results[lang] = {
                'accuracy': lang_acc,
                'samples': len(lang_indices)
            }
    
    return accuracy, all_preds, all_labels, all_probs, per_lang_results

def run_codet5_experiment(seed, device, tag_vocab, train_data, test_data, 
                         train_encodings, test_encodings, save_results=True):
    """Run a single CodeT5 experiment with given seed"""
    print(f"\n{'='*80}")
    print(f"CODET5 EXPERIMENT WITH SEED: {seed}")
    print(f"{'='*80}")
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    label_encoder = LabelEncoder()
    label_encoder.fit(tag_vocab)
    
    train_dataset = CodeDataset(
        train_encodings,
        label_encoder.transform(train_data["language"])
    )
    
    test_dataset = CodeDataset(
        test_encodings,
        label_encoder.transform(test_data["language"])
    )
    
    num_classes = len(tag_vocab)
    model = CodeT5Classifier(num_classes, MODEL_NAME, device, freeze_layers=True)
    
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=3e-5,
        weight_decay=0.01
    )
    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=3, 
        eta_min=1e-6
    )
    
    criterion = nn.CrossEntropyLoss()
    
    batch_size = 32  
    gradient_accumulation_steps = 1 
    effective_batch_size = batch_size
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    print(f"\nTraining configuration:")
    print(f"  Seed: {seed}")
    print(f"  Model: {MODEL_NAME} (Encoder only)")
    print(f"  Batch size: {batch_size}")
    print(f"  Learning rate: {3e-5}")
    print(f"  Training samples: {len(train_dataset)}")
    print(f"  Test samples: {len(test_dataset)}")
    
    print(f"\n{'='*50}")
    print(f"Training for seed {seed}")
    print(f"{'='*50}")
    
    epochs = 3
    best_accuracy = 0
    training_history = []
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 40)
        
        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, criterion, device, gradient_accumulation_steps
        )
        
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        
        val_accuracy, val_preds, val_labels, val_probs, val_per_lang = evaluate_model(
            model, test_loader, device, tag_vocab
        )
        
        print(f"Epoch {epoch+1} Summary:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Validation Accuracy: {val_accuracy:.4f}, LR: {current_lr:.2e}")
        
        training_history.append({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_acc': val_accuracy,
            'learning_rate': current_lr
        })
        
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_epoch = epoch + 1
            best_model_state = model.state_dict().copy()
            print(f"  ✓ New best validation accuracy!")
    
    print(f"\n{'='*40}")
    print(f"Final Evaluation for seed {seed}")
    print(f"{'='*40}")
    
    model.load_state_dict(best_model_state)
    model.eval()
    
    final_accuracy, all_preds, all_labels, all_probs, per_lang_results = evaluate_model(
        model, test_loader, device, tag_vocab
    )
    
    confidences = np.max(all_probs, axis=1)
    
    print(f"\n✓ Experiment completed for seed {seed}!")
    print(f"  Best Epoch: {best_epoch}")
    print(f"  Final Test Accuracy: {final_accuracy:.4f}")
    print(f"  Best Training Accuracy: {best_train_acc:.4f}")
    print(f"  Best Training Loss: {best_train_loss:.4f}")
    print(f"  Average Confidence: {np.mean(confidences):.4f}")
    
    if save_results:
        model_save_path = f"/home/aman_swaraj/Downloads/Codelite/codet5_seed{seed}.pth"
        torch.save({
            'seed': seed,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'final_accuracy': final_accuracy,
            'train_accuracy': best_train_acc,
            'train_loss': best_train_loss,
            'best_epoch': best_epoch,
            'tag_vocab': tag_vocab,
            'label_encoder_classes': label_encoder.classes_.tolist(),
            'model_name': MODEL_NAME,
            'training_history': training_history
        }, model_save_path)
        print(f"✓ Model saved to {model_save_path}")
        
        predictions_df = pd.DataFrame({
            'true_label': [tag_vocab[l] for l in all_labels],
            'predicted_label': [tag_vocab[p] for p in all_preds],
            'confidence': confidences,
            'is_correct': [1 if p == l else 0 for p, l in zip(all_preds, all_labels)]
        })
        
        results_path = f"/home/aman_swaraj/Downloads/Codelite/codet5_results_seed{seed}.csv"
        predictions_df.to_csv(results_path, index=False)
        print(f"✓ Results saved to {results_path}")
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        'seed': seed,
        'final_accuracy': final_accuracy,
        'train_accuracy': best_train_acc,
        'train_loss': best_train_loss,
        'best_epoch': best_epoch,
        'per_language_accuracy': per_lang_results,
        'predictions': all_preds,
        'labels': all_labels,
        'confidences': confidences,
        'training_history': training_history
    }

def main():
    print("\n" + "=" * 80)
    print("MULTI-SEED CODET5 EXPERIMENT")
    print("=" * 80)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory: {gpu_memory:.2f} GB")
        
        torch.cuda.empty_cache()
    
    seeds = [42, 123, 456, 789, 999]  
    print(f"\nRunning experiments for {len(seeds)} seeds: {seeds}")
    
    all_results = []
    
    for i, seed in enumerate(seeds):
        print(f"\n{'#'*80}")
        print(f"Experiment {i+1}/{len(seeds)}")
        print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'#'*80}")
        
        try:
            result = run_codet5_experiment(
                seed=seed,
                device=device,
                tag_vocab=tag_vocab,
                train_data=train_data,
                test_data=test_data,
                train_encodings=train_encodings,
                test_encodings=test_encodings,
                save_results=True
            )
            
            all_results.append(result)
            
            print(f"\n✓ Experiment {i+1} completed successfully!")
            
        except Exception as e:
            print(f"\n✗ Experiment {i+1} failed with error:")
            import traceback
            print(f"  Error type: {type(e).__name__}")
            print(f"  Error message: {str(e)}")
            print(f"  Traceback (simplified):")
            tb_lines = traceback.format_exc().split('\n')
            for line in tb_lines[-10:]:  
                if line.strip():
                    print(f"    {line}")
            print(f"  Skipping seed {seed}...")
            continue
    
    if not all_results:
        print("\nNo experiments completed successfully!")
        return
    
    print("\n" + "=" * 80)
    print("SUMMARY OF ALL CODET5 EXPERIMENTS")
    print("=" * 80)
    
    final_accuracies = [r['final_accuracy'] for r in all_results]
    train_accuracies = [r['train_accuracy'] for r in all_results]
    train_losses = [r['train_loss'] for r in all_results]
    best_epochs = [r['best_epoch'] for r in all_results]
    seeds_list = [r['seed'] for r in all_results]
    
    summary_data = []
    for i, result in enumerate(all_results):
        summary_data.append({
            'Seed': result['seed'],
            'Best_Epoch': result['best_epoch'],
            'Final_Accuracy': result['final_accuracy'],
            'Final_Accuracy_%': f"{result['final_accuracy']*100:.2f}%",
            'Train_Accuracy': result['train_accuracy'],
            'Train_Loss': f"{result['train_loss']:.4f}"
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    if len(final_accuracies) > 0:
        mean_accuracy = np.mean(final_accuracies)
        std_accuracy = np.std(final_accuracies)
        min_accuracy = np.min(final_accuracies)
        max_accuracy = np.max(final_accuracies)
    else:
        mean_accuracy = std_accuracy = min_accuracy = max_accuracy = 0
    
    print(f"\nOVERALL STATISTICS:")
    print(f"  Number of experiments: {len(all_results)}")
    print(f"  Mean Final Accuracy: {mean_accuracy:.4f} ({mean_accuracy*100:.2f}%)")
    print(f"  Std Final Accuracy: {std_accuracy:.4f}")
    print(f"  Min Final Accuracy: {min_accuracy:.4f} ({min_accuracy*100:.2f}%)")
    print(f"  Max Final Accuracy: {max_accuracy:.4f} ({max_accuracy*100:.2f}%)")
    print(f"  Range: {max_accuracy - min_accuracy:.4f}")
    
    print(f"\nDETAILED RESULTS:")
    print(summary_df.to_string(index=False))
    
    if len(all_results) > 0:
        print(f"\n{'='*80}")
        print("LANGUAGE-WISE ANALYSIS (Across all seeds)")
        print(f"{'='*80}")
        
        lang_accuracies = {lang: [] for lang in tag_vocab}
        lang_samples = {lang: [] for lang in tag_vocab}
        
        for result in all_results:
            for lang, lang_data in result['per_language_accuracy'].items():
                lang_accuracies[lang].append(lang_data['accuracy'])
                lang_samples[lang].append(lang_data['samples'])
        
        lang_summary = []
        for lang in tag_vocab:
            if lang_accuracies[lang]:
                mean_acc = np.mean(lang_accuracies[lang])
                std_acc = np.std(lang_accuracies[lang])
                min_acc = np.min(lang_accuracies[lang])
                max_acc = np.max(lang_accuracies[lang])
                avg_samples = np.mean(lang_samples[lang]) if lang_samples[lang] else 0
                
                lang_summary.append({
                    'Language': lang,
                    'Mean_Accuracy': f"{mean_acc:.4f}",
                    'Std_Accuracy': f"{std_acc:.4f}",
                    'Min_Accuracy': f"{min_acc:.4f}",
                    'Max_Accuracy': f"{max_acc:.4f}",
                    'Range': f"{max_acc - min_acc:.4f}",
                    'Avg_Samples': int(avg_samples)
                })
        
        if lang_summary:
            lang_summary.sort(key=lambda x: float(x['Mean_Accuracy']), reverse=True)
            lang_df = pd.DataFrame(lang_summary)
            
            print("\nLanguage-wise performance across seeds (sorted by mean accuracy):")
            print(lang_df.to_string(index=False))
            
            lang_mean_accuracies = [float(lang['Mean_Accuracy']) for lang in lang_summary]
            print(f"\nLanguage-wise Statistics:")
            print(f"  Mean accuracy across languages: {np.mean(lang_mean_accuracies):.4f}")
            print(f"  Std of language accuracies: {np.std(lang_mean_accuracies):.4f}")
            print(f"  Best performing language: {lang_summary[0]['Language']} ({lang_summary[0]['Mean_Accuracy']})")
            print(f"  Worst performing language: {lang_summary[-1]['Language']} ({lang_summary[-1]['Mean_Accuracy']})")
    
    

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime

base_path = "/home/aman_swaraj/Downloads/Codelite"
codet5_summary_files = sorted(glob.glob(f"{base_path}/codet5_multiseed_summary_*.csv"))
codet5_lang_files = sorted(glob.glob(f"{base_path}/codet5_lang_summary_*.csv"))
codet5_stats_files = sorted(glob.glob(f"{base_path}/codet5_stats_summary_*.csv"))

print("=" * 80)
print("CODET5 MULTI-SEED EXPERIMENT RESULTS")
print("=" * 80)

if codet5_summary_files:
    latest_summary = codet5_summary_files[-1]
    print(f"\n📊 Loading summary from: {os.path.basename(latest_summary)}")
    
    summary_df = pd.read_csv(latest_summary)
    
    print(f"\n{'='*60}")
    print("EXPERIMENT SUMMARY")
    print(f"{'='*60}")
    print(f"Number of experiments: {len(summary_df)}")
    print(f"Seeds: {list(summary_df['seed'])}")
    
    mean_acc = summary_df['final_accuracy'].mean()
    std_acc = summary_df['final_accuracy'].std()
    min_acc = summary_df['final_accuracy'].min()
    max_acc = summary_df['final_accuracy'].max()
    
    print(f"\n📈 OVERALL ACCURACY STATISTICS:")
    print(f"  Mean: {mean_acc:.4f} ({mean_acc*100:.2f}%)")
    print(f"  Std: {std_acc:.4f}")
    print(f"  Min: {min_acc:.4f} ({min_acc*100:.2f}%)")
    print(f"  Max: {max_acc:.4f} ({max_acc*100:.2f}%)")
    print(f"  Range: {max_acc - min_acc:.4f}")
    
    print(f"\n📋 DETAILED RESULTS:")
    display_df = summary_df[['seed', 'best_epoch', 'final_accuracy', 'train_accuracy', 'train_loss']].copy()
    display_df['final_accuracy'] = display_df['final_accuracy'].apply(lambda x: f"{x:.4f}")
    display_df['train_accuracy'] = display_df['train_accuracy'].apply(lambda x: f"{x:.4f}")
    display_df['train_loss'] = display_df['train_loss'].apply(lambda x: f"{x:.4f}")
    display_df.columns = ['Seed', 'Best Epoch', 'Final Acc', 'Train Acc', 'Train Loss']
    print(display_df.to_string(index=False))
    
    if codet5_lang_files:
        latest_lang = codet5_lang_files[-1]
        print(f"\n🌍 Loading language-wise summary from: {os.path.basename(latest_lang)}")
        
        lang_df = pd.read_csv(latest_lang)
        
        print(f"\n{'='*60}")
        print("LANGUAGE-WISE PERFORMANCE (Sorted by Mean Accuracy)")
        print(f"{'='*60}")
        
        print(f"\n🏆 TOP 5 LANGUAGES:")
        top_languages = lang_df.head(5)
        for idx, row in top_languages.iterrows():
            print(f"  {idx+1}. {row['Language']:15s}: {row['Mean_Accuracy']} ± {row['Std_Accuracy']}")
        
        print(f"\n📉 BOTTOM 5 LANGUAGES:")
        bottom_languages = lang_df.tail(5)
        for i, (idx, row) in enumerate(bottom_languages.iterrows(), 1):
            print(f"  {i}. {row['Language']:15s}: {row['Mean_Accuracy']} ± {row['Std_Accuracy']}")
    
    if codet5_stats_files:
        latest_stats = codet5_stats_files[-1]
        stats_df = pd.read_csv(latest_stats)
        
        print(f"\n{'='*60}")
        print("EXPERIMENT STATISTICS")
        print(f"{'='*60}")
        print(f"Timestamp: {stats_df.iloc[0]['timestamp']}")
        print(f"Model: {stats_df.iloc[0]['model_name']}")
        print(f"Training samples: {stats_df.iloc[0]['training_samples']:,}")
        print(f"Test samples: {stats_df.iloc[0]['test_samples']:,}")
        print(f"Number of languages: {stats_df.iloc[0]['num_languages']}")
    
    print(f"\n{'='*60}")
    print("INDIVIDUAL SEED RESULTS")
    print(f"{'='*60}")
    
    for seed in summary_df['seed']:
        result_file = f"{base_path}/codet5_results_seed{seed}.csv"
        if os.path.exists(result_file):
            pred_df = pd.read_csv(result_file)
            accuracy = pred_df['is_correct'].mean()
            avg_confidence = pred_df['confidence'].mean()
            print(f"Seed {seed}: Accuracy = {accuracy:.4f}, Avg Confidence = {avg_confidence:.4f}")
    
    print(f"\n{'='*60}")
    print("CONFIDENCE ANALYSIS")
    print(f"{'='*60}")
    
    confidences = []
    accuracies = []
    for seed in summary_df['seed']:
        result_file = f"{base_path}/codet5_results_seed{seed}.csv"
        if os.path.exists(result_file):
            pred_df = pd.read_csv(result_file)
            confidences.append(pred_df['confidence'].mean())
            accuracies.append(pred_df['is_correct'].mean())
    
    if len(confidences) > 1:
        correlation = np.corrcoef(confidences, accuracies)[0, 1]
        print(f"Correlation between confidence and accuracy: {correlation:.4f}")
    
    print(f"\n{'='*60}")
    print("PERFORMANCE SUMMARY")
    print(f"{'='*60}")
    
    best_idx = summary_df['final_accuracy'].idxmax()
    worst_idx = summary_df['final_accuracy'].idxmin()
    
    best_seed = summary_df.iloc[best_idx]['seed']
    best_acc = summary_df.iloc[best_idx]['final_accuracy']
    worst_seed = summary_df.iloc[worst_idx]['seed']
    worst_acc = summary_df.iloc[worst_idx]['final_accuracy']
    
    print(f"✅ Best Seed: {best_seed} (Accuracy: {best_acc:.4f})")
    print(f"❌ Worst Seed: {worst_seed} (Accuracy: {worst_acc:.4f})")
    print(f"📊 Overall Mean: {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"🎯 Range: {(max_acc - min_acc):.4f}")
    
    variability = (std_acc / mean_acc) * 100 if mean_acc > 0 else 0
    print(f"📏 Coefficient of Variation: {variability:.2f}%")
    
    print(f"\n{'='*60}")
    print("NEXT STEPS")
    print(f"{'='*60}")
    print("1. To run all 5 seeds, change 'seeds = [42, 123]' to 'seeds = [42, 123, 456, 789, 999]'")
    print("2. Check individual seed results in:")
    print(f"   - {base_path}/codet5_results_seed*.csv")
    print("3. Models saved as:")
    print(f"   - {base_path}/codet5_seed*.pth")

else:
    print("❌ No CodeT5 summary files found!")
    print("\nCheck if the experiments completed successfully.")
    print("Available files:")
    all_files = glob.glob(f"{base_path}/codet5_*")
    for f in all_files[:10]:  
        print(f"  - {os.path.basename(f)}")
    if len(all_files) > 10:
        print(f"  ... and {len(all_files) - 10} more")